# 04 – Exploring HealthSynth with DuckDB

## How quickly can we answer commercial questions using SQL?

HealthSynth can export every generated dataset directly into a DuckDB database.

DuckDB is an embedded analytical database that requires no server installation and works well for local data exploration.

This means we can move directly from simulation to SQL analysis in just a few seconds.

In this notebook we'll:

- generate a market
- connect to DuckDB
- inspect the available tables
- answer a few business questions using SQL

No previous DuckDB experience is required.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

from healthsynth.generator import generate

## Generate a Small Market

We'll create a small simulation so that we can focus on querying the generated data rather than waiting for the simulation to finish.

In [ ]:
output_dir = Path("../output/notebook_04_duckdb")

generate(
    config_path="../configs/profiles/oncology_training.yaml",
    hcps=500,
    output_dir=str(output_dir),
    output_format="all",
)

## Connect to the DuckDB Database

HealthSynth exports a DuckDB database alongside the CSV files.

Let's open it.

In [ ]:
database_path = next(output_dir.glob("*.duckdb"))

conn = duckdb.connect(str(database_path))

## What Tables Are Available?

Before writing SQL, it's useful to discover what data exists.

In [ ]:
conn.execute(
    "SHOW TABLES"
).df()

## Business Question 1

### Which products generated the highest number of new prescriptions?

In [ ]:
conn.execute("""
SELECT
    p.product_name,
    SUM(rx.nrx) AS total_nrx
FROM prescriptions rx
JOIN product p
ON rx.product_id = p.product_id
GROUP BY
    p.product_name
ORDER BY total_nrx DESC
""").df()

This query joins the product master with prescription data to calculate total new prescriptions (NRx) by product.

Although the data is synthetic, the workflow is very similar to how commercial analytics teams analyse pharmaceutical data.

## Business Question 2

### Which specialties received the most promotional activity?

In [ ]:
conn.execute("""
SELECT
    specialty,
    COUNT(*) AS calls
FROM call_activity
GROUP BY specialty
ORDER BY calls DESC
""").df()

## Business Question 3

### How did market demand change over time?

In [ ]:
market_demand = conn.execute("""
SELECT
    month,
    market_nrx
FROM market_demand
ORDER BY month
""").df()

market_demand

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))

plt.plot(
    market_demand["month"],
    market_demand["market_nrx"],
    marker="o",
)

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()

Demand represents the total prescription opportunity available in the simulated market.

Later notebooks show how commercial events redistribute that demand among competing products.

## Why DuckDB?

DuckDB is particularly useful because it allows us to:

- analyse data using SQL
- build dashboards
- prototype analytics
- integrate with Python
- avoid installing a database server

For local learning and experimentation, it provides a lightweight analytical environment.

## Key Takeaways

In this notebook we learned that:

- HealthSynth exports DuckDB databases automatically
- DuckDB makes SQL analysis immediately available
- generated pharmaceutical datasets can be queried using standard SQL
- business questions often require joining multiple datasets rather than analysing tables in isolation

DuckDB provides an easy bridge between simulation and analytics.

## Try It Yourself

Try writing a few additional SQL queries.

For example:

- Which territory generated the most prescriptions?
- How many calls occurred each month?
- Which products have the highest average market share?
- Which HCP specialty generated the highest NRx?

Can you answer each question using only SQL?

As you progress through the remaining notebooks, revisit these queries after introducing commercial events such as product launches, loss of exclusivity, competitor entry, and market access. Observe how the answers change.

## What's Next?

We now know how to explore HealthSynth data using SQL.

In Notebook 05 we'll begin analysing one of the most important commercial concepts in pharmaceutical analytics: **market share**.

We'll learn how products compete for demand and why market share changes over time.